# Simulating revenue impacts of TCJA extension with Tax-Brain and OG-USA

In [43]:
# imports
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition
from taxbrain import TaxBrain
import os
import numpy as np

In [27]:
reform_url = "https://raw.githubusercontent.com/PSLmodels/Tax-Calculator/master/taxcalc/reforms/ext.json"
# Tax-Brain static revenue estimate
tb_ext = TaxBrain(2026, 2034, microdata="TMD", reform=reform_url, stacked=False)
tb_ext.run()
tb_ext.weighted_totals("combined", include_total=True) * 1e-12

,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.504047,4.691042,4.882284,5.087651,5.299603,5.519068,5.745442,5.976296,6.212457,47.917890
Reform,4.220850,4.399709,4.582886,4.779384,4.982565,5.193152,5.410631,5.632493,5.859609,45.061278
Difference,-0.283197,-0.291332,-0.299398,-0.308267,-0.317038,-0.325917,-0.334811,-0.343803,-0.352849,-2.856612


In [28]:
# Tax-Brain micro dynamic revenue estimate
tb_ext_br = TaxBrain(2026, 2034, microdata="TMD", reform=reform_url, stacked=False, behavior={"sub": 0.25})
tb_ext_br.run()
# tb_ext_br.stacked_table * 1e-12
tb_ext_br.weighted_totals("combined", include_total=True) * 1e-12

,2026,2027,2028,2029,2030,2031,2032,2033,2034,Total
Base,4.504047,4.691042,4.882284,5.087651,5.299603,5.519068,5.745442,5.976296,6.212457,47.917890
Reform,4.277884,4.459179,4.644816,4.843943,5.049837,5.262878,5.483251,5.707902,5.938190,45.667881
Difference,-0.226163,-0.231863,-0.237468,-0.243707,-0.249766,-0.256190,-0.262190,-0.268395,-0.274267,-2.250009


In [30]:
# Read in OG-USA results
CUR_DIR = os.getcwd()
sim_path = os.path.join(CUR_DIR, "TCJA_Extension")
base_params = safe_read_pickle(os.path.join(sim_path, "OUTPUT_BASELINE", "model_params.pkl"))
base_tpi = safe_read_pickle(os.path.join(sim_path, "OUTPUT_BASELINE", "TPI", "TPI_vars.pkl"))
base_ss = safe_read_pickle(os.path.join(sim_path, "OUTPUT_BASELINE", "SS", "SS_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(sim_path, "OUTPUT_REFORM", "model_params.pkl"))
reform_tpi = safe_read_pickle(os.path.join(sim_path, "OUTPUT_REFORM", "TPI", "TPI_vars.pkl"))
reform_ss = safe_read_pickle(os.path.join(sim_path, "OUTPUT_REFORM", "SS", "SS_vars.pkl"))

In [31]:
# Do dynamic revenue decomposition -- this will give percentage changes in revenue
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2026, num_years=9, full_break_out=True)
df

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034,SS
0,IIT: Pct Change due to tax rates,-7.75,-7.76,-7.76,-7.76,-7.77,-7.78,-7.78,-7.78,-7.78,-7.77,-7.75
1,IIT: Pct Change due to behavior,1.37,1.35,1.33,1.32,1.31,1.31,1.30,1.30,1.30,1.32,1.30
2,IIT: Pct Change due to macro,-0.02,-0.05,-0.08,-0.11,-0.14,-0.18,-0.21,-0.24,-0.28,-0.15,0.22
3,IIT: Overall Pct Change in taxes,-6.51,-6.56,-6.61,-6.65,-6.69,-6.74,-6.78,-6.81,-6.85,-6.69,-6.35
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,0.83,0.84,0.84,0.83,0.81,0.79,0.76,0.72,0.68,0.79,1.83
6,CIT: Pct Change due to macro,0.43,0.34,0.28,0.23,0.19,0.17,0.16,0.16,0.17,0.24,-1.00
7,CIT: Overall Pct Change in taxes,1.27,1.19,1.12,1.06,1.01,0.96,0.92,0.89,0.85,1.03,0.81
8,All: Pct Change due to tax rates,-7.25,-7.25,-7.25,-7.26,-7.26,-7.27,-7.27,-7.27,-7.27,-7.26,-7.24
9,All: Pct Change due to behavior,1.33,1.32,1.30,1.29,1.28,1.27,1.26,1.26,1.25,1.28,1.34


In [53]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 9), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.39,-0.42,-0.43,-0.45,-0.46,-0.48,-0.50,-0.52,-0.54,-4.20
9,Rev Change Due to Behavior,0.07,0.08,0.08,0.08,0.08,0.08,0.09,0.09,0.09,0.74
10,Rev Change Due to Macro,0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.02,-0.02,-0.07
11,Total Revenue Change,-0.32,-0.35,-0.36,-0.38,-0.39,-0.42,-0.43,-0.45,-0.47,-3.58


In [64]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 9), (df_levels.shape[0], 1)) / 100
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.36,-0.39,-0.40,-0.42,-0.43,-0.45,-0.47,-0.48,-0.50,-3.91
1,Rev Change Due to Behavior,0.06,0.07,0.07,0.07,0.07,0.08,0.08,0.08,0.08,0.66
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.02,-0.02,-0.08
3,Total Revenue Change,-0.30,-0.33,-0.34,-0.36,-0.37,-0.39,-0.41,-0.42,-0.44,-3.37


In [73]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = (tb_ext.weighted_totals("combined", include_total=True) * 1e-12).loc["Base", [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034]].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 9), (df_levels.shape[0], 1)) / 100
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.35,-0.36,-0.38,-0.39,-0.41,-0.43,-0.45,-0.47,-0.48,-3.72
1,Rev Change Due to Behavior,0.06,0.06,0.07,0.07,0.07,0.07,0.07,0.08,0.08,0.63
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.01,-0.01,-0.01,-0.01,-0.01,-0.02,-0.07
3,Total Revenue Change,-0.29,-0.31,-0.32,-0.34,-0.35,-0.37,-0.39,-0.41,-0.43,-3.21


# NOTE:

The revenue impact from OG-USA is larger, even when applying the percentage change to the Tax-Calculator baseline (here, -3.21T from OG-USA vs a static Tax-Brain estimate of -2.85T and a micro-dynamic of -2.25T).

This could be due to the fact that OG-USA tax functions are approximations.  If we want consistency, we can apply the OG-USA behavior and macro percentage changes to the static Tax-Brain results.

## Example:

In [80]:
df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = (tb_ext.weighted_totals("combined", include_total=True) * 1e-12).loc["Difference", [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034]].values
tc_reform = (tb_ext_br.weighted_totals("combined", include_total=True) * 1e-12).loc["Reform", [2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034]].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2026-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2026,2027,2028,2029,2030,2031,2032,2033,2034,2026-2034
0,Rev Change Due to Tax Rates,-0.28,-0.29,-0.30,-0.31,-0.32,-0.33,-0.33,-0.34,-0.35,-2.86
1,Rev Change Due to Behavior,0.06,0.06,0.06,0.06,0.07,0.07,0.07,0.07,0.08,0.60
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
3,Total Revenue Change,-0.22,-0.23,-0.24,-0.24,-0.25,-0.26,-0.26,-0.27,-0.28,-2.26


In the end the Tax-Brain behavior is similar to the OG-USA behavior and macro effects (changes to r and w) are not super important over this time horizon.